# Cell Segmentation Pipeline

## Overview
This notebook performs automated cell segmentation on fluorescence microscopy images using intensity-based thresholding and morphological operations.

## Input/Output
- **Input**: `*_all_SUM.tif` files (intensity sum images of cells)
- **Output**: `*_ROI.tif` files (binary masks representing segmented cell regions)

## Configuration Parameters
- `median_size`: 2 (kernel size for noise reduction filter)
- `area_threshold`: 100 (maximum hole size to fill, in pixels)
- `min_size`: 10,000 (minimum object size to keep, in pixels)
- Thresholding: Li's automatic method (adaptive)

## Important Notes
⚠️ **Manual intervention required**: Cells that are touching each other need to be manually separated by drawing black lines between them in the "_all_SUM" images before running this segmentation.

## Processing Steps
1. Load intensity sum images
2. Apply median filter for noise reduction
3. Apply Li's automatic thresholding
4. Fill small holes within objects
5. Remove small objects (noise/debris)
6. Save binary masks as ROI files

In [1]:
# Load all required libraris
import os
import numpy as np
from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.filters import threshold_li
from skimage.io import imread, imsave
from scipy.ndimage import median_filter

## Define segmentation function

In [9]:
def process_image(file_path, output_path):
    """
    Process a single image for cell segmentation.
    
    Parameters:
    -----------
    file_path : str
        Path to input image file
    output_path : str 
        Path where processed binary mask will be saved
        
    Processing steps:
    1. Load image
    2. Apply median filter for noise reduction
    3. Apply Li's automatic threshold method
    4. Fill small holes and remove small objects
    5. Save binary mask
    """
    
    # Load the input image (should be grayscale intensity sum)
    img = imread(file_path)
    
    # Apply median filter with kernel size 2x2 for noise reduction
    # This helps smooth the image while preserving edges
    img_blur = median_filter(img, size=2)
    
    # Apply Li's automatic thresholding method
    # Li's method works well for images with bimodal intensity distributions
    # (background vs foreground/cells)
    thresh_value = threshold_li(img_blur)
    binary_mask = img_blur > thresh_value
    
    # Post-processing to clean up the binary mask:
    
    # Fill small holes (up to 100 pixels) within detected objects
    # This removes small dark spots inside cells
    binary_mask = remove_small_holes(binary_mask, area_threshold=100)
    
    # Remove small objects (smaller than 20,000 pixels)
    # This filters out noise and debris, keeping only cell-sized objects
    # Note: 10,000 px might be quite large - consider if this threshold is appropriate
    binary_mask = remove_small_objects(binary_mask, min_size=20000)
    
    # Convert boolean mask to 8-bit image (0-255 range)
    # True pixels become 255 (white), False pixels become 0 (black)
    binary_mask = (binary_mask * 255).astype(np.uint8)
    
    # Save the processed binary mask
    imsave(output_path, binary_mask)


## Batch processing section

In [10]:
# Define input and output directories
input_folder = "C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/"
output_folder = "C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/"   # Same folder for output

# Create output directory if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Process all matching files in the input folder 
for file_name in os.listdir(input_folder):
    # Only process files ending with "_all_SUM.tif"
    if file_name.endswith("_all_SUM.tif"):
        # Construct full file paths
        input_path = os.path.join(input_folder, file_name)
        # Generate output filename by replacing suffix
        output_path = os.path.join(output_folder, file_name.replace("_all_SUM.tif", "_ROI.tif"))
        
        # Process the image
        process_image(input_path, output_path)
        print(f"Processed and saved: {output_path}")


Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_64_ROI.tif
Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_66_ROI.tif
Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_68_ROI.tif
Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_70_ROI.tif
Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_72_ROI.tif
Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_74_ROI.tif
Processed and saved: C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA_1-5/low_FRET_1-5_76_ROI.tif
